<a href="https://colab.research.google.com/github/aehaf988/Pro1/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
#import and file paths
import math
import pandas as pd
import numpy as np
from scipy.special import gammaln
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt

signature = '3' #large deletions with microhomology at breakpoints
#parameter: theta = SBS3 weight

In [24]:

RAW_URL = 'https://raw.githubusercontent.com/aehaf988/Pro1/main/Breast_genomes_mutational_catalog_96_subs_SBS3.txt'
catalog = pd.read_csv(RAW_URL, sep='\t', index_col=0, decimal=',')

print('Shape:', catalog.shape)
print('Dtypes:', catalog.dtypes.head())

# --- THE MISSING LINE ---
ref    = catalog['SBS3']
counts = catalog.drop(columns='SBS3')

print('Total mutations (samples only):', int(counts.values.sum()))
print('First mutation types:', list(catalog.index[:3]))
print('First samples:', list(counts.columns[:3]))     # note: counts, not catalog



Shape: (96, 120)
Dtypes: SBS3       object
PD3851a     int64
PD3890a     int64
PD3904a     int64
PD3905a     int64
dtype: object
Total mutations (samples only): 647692
First mutation types: ['A[C>A]A', 'A[C>A]C', 'A[C>A]G']
First samples: ['PD3851a', 'PD3890a', 'PD3904a']


In [25]:

def poisson_log_pmf(k, lam):
    return k * math.log(lam) - lam - math.lgamma(k + 1)

def NLL(theta, K_obs, N_total):
    lam = N_total * theta
    return -poisson_log_pmf(K_obs, lam)

K_obs   = 29
sample  = counts.columns[0]              # e.g. 'PD3851a'
k = counts[sample].values.astype(float)   # (96,) ← this is what was missing
N_total = 1000

result = minimize_scalar(NLL, bounds=(1e-9, 1.0),
                         method="bounded",
                         args=(K_obs, N_total))
print("theta =", result.x, "  -loglik =", result.fun)

def poisson_log_pmf(k, lam):
    """Vectorized log Poisson PMF. Works for scalar or array k, lam."""
    k   = np.asarray(k, dtype=float)
    lam = np.asarray(lam, dtype=float)
    log_lam = np.where(lam > 0, np.log(np.maximum(lam, 1e-300)), -np.inf)
    return k * log_lam - lam - gammaln(k + 1)

def NLL_signature(theta, k_obs, r, N_total):
    """Whole 96-channel fit against a single reference signature r."""
    k_obs = np.asarray(k_obs, dtype=float)
    r     = np.asarray(r,     dtype=float)
    lam   = N_total * theta * r
    return -poisson_log_pmf(k_obs, lam).sum()

r = (
    catalog['SBS3']
    .astype(str).str.replace(',', '.', regex=False)
    .astype(float).values
)
r = r / r.sum()                             # normalize
counts = catalog.drop(columns='SBS3').astype(int)

sample  = counts.columns[0]                 # e.g. 'PD3851a'
k       = counts[sample].values.astype(float)   # (96,)
N_total = int(k.sum())                      # per-sample total

print("ready:", sample, " N_total =", N_total,
      " k.shape =", k.shape, " r.shape =", r.shape, " r.sum() =", r.sum())

print("ready:", sample, "N_total =", N_total, "k.shape =", k.shape, "r.shape =", r.shape)





theta = 0.028999015183098803   -loglik = 2.6054599142824344
ready: PD3851a  N_total = 1561  k.shape = (96,)  r.shape = (96,)  r.sum() = 1.0
ready: PD3851a N_total = 1561 k.shape = (96,) r.shape = (96,)


In [26]:
def poisson_log_pmf(k, lam):
    k   = np.asarray(k, dtype=float)
    lam = np.asarray(lam, dtype=float)
    log_lam = np.where(lam > 0, np.log(np.maximum(lam, 1e-300)), -np.inf)
    return k * log_lam - lam - gammaln(k + 1)

def NLL_signature(theta, k_obs, r, N_total):
    k_obs = np.asarray(k_obs, dtype=float)
    r     = np.asarray(r,     dtype=float)
    lam   = N_total * theta * r
    return -poisson_log_pmf(k_obs, lam).sum()

In [27]:
# Split once, right after loading
ref    = catalog['SBS3'].astype(str).str.replace(',', '.').astype(float).values
counts = catalog.drop(columns='SBS3').astype(int)

# Now the total is over actual observations only
N_total = int(counts.values.sum())
print("N_total (observed mutations across all samples):", N_total)


def simulate(theta, N_total, r, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    r = np.asarray(r, float)
    r = r / r.sum()
    lam = N_total * theta * r          # (96,) expected counts
    return rng.poisson(lam)            # (96,) simulated counts

rng   = np.random.default_rng(0)
k_sim = simulate(theta=1.0, N_total=N_total, r=ref, rng=rng)
print("simulated counts:", k_sim[:5], "  sum:", k_sim.sum())


N_total (observed mutations across all samples): 647692
simulated counts: [13595 10321  1181  7955 12948]   sum: 649142


In [28]:
#slope

def slope_at(score, theta, h):
  return (score(theta+h) -score(theta-h)) / (2*h)

K = k.sum()
N = N_total

def score(theta):
    return NLL_signature(theta, k, r, N)

for theta in [0.5, 1.0, 1.5]:
    num = slope_at(score, theta, h=1e-5)
    ana = N - K / theta
    print(f"theta={theta}  numeric={num:.6f}  analytic={ana:.6f}  "
          f"diff={num-ana:.2e}")

theta=0.5  numeric=644569.999995  analytic=644570.000000  diff=-4.71e-06
theta=1.0  numeric=646130.999993  analytic=646131.000000  diff=-7.40e-06
theta=1.5  numeric=646651.333332  analytic=646651.333333  diff=-1.50e-06


In [ ]:
# uncertainty: parametric bootstrap for one sample
A_LCG = 1664525
C_LCG = 1013904223
M_LCG = 2**32


def lcg (state, a, c, m):
  return (a*state+c)%m

def uniforms(k, seed):
    out = [0.0] * k
    state = seed
    for i in range(k):
        state = lcg(state, A_LCG, C_LCG, M_LCG)   # ← 4 args
        out[i] = state / M_LCG
    return out

def choose(r, probs):
  running = 0.0
  for k in range(len(probs)):
    running = running + probs[k]
    if r < running:
      return k
  return len(probs) - 1

B = 1000          # number of bootstrap resamples
SEED = 8

rs = uniforms(B * N, SEED) #every draw the bootstrap needs, '40' per sample
equal = [1/N] * N

boot_estimates = [0.0] * B
for b in range(B):
  resample = N * [0.0]
  for i in range(N):
    j = choose(rs[b * N + i], equal)
    resample[i] = counts[j]
  boot_estimates[b] = mean(resample)

sigma_boot = std(boot_estimates)


In [ ]:
#plots
#a. distribution of a signature across samples
#b. observed vs fitted spectrum for one sample
#c. per-signature weight across sample with CIs